
---
## Section 2 - Datasets, Feature Engineering & Experimental Design

### 2.1 Dataset Overview - Three VCT Seasons

The B-Rank mission requires three datasets. We utilise three chronological slices of VCT professional match data, each representing a distinct competitive season. Using temporal slicing instead of arbitrary dataset fragmentation is methodologically sound - it mirrors how a real predictive system would be trained (historical data) and evaluated (future matches).

| Dataset | VCT Season | Matches | Description |
|---------|-----------|---------|-------------|
| **D1** | 2023 | 330 | Inaugural VCT franchise season. Americas, EMEA, Pacific leagues + World Champions 2023. |
| **D2** | 2024 | 434 | Second franchise season. Added China region. Masters Madrid, Shanghai, Champions 2024. |
| **D3** | 2025 | 501 | Current season (test set). Kickoffs, Stage 1 & 2, Masters Bangkok, Toronto, Champions 2025. |

**Experimental Design:** We train on D1 ∪ D2 (764 matches) and evaluate on D3 (501 matches). This **chronological holdout** is the only valid split for temporal sports data - using random k-fold cross-validation would constitute data leakage (future matches would appear in training folds).

### 2.2 Feature Engineering Note

The 13 features are **pre-match** features only - computed exclusively from data available **before** the match is played. In-match statistics (avg_rating per match, ACS, kills) are used only to **update** the rolling historical averages for future matches, never as direct model inputs. This strict temporal discipline prevents any form of data leakage.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, classification_report
)
import xgboost as xgb

# Aesthetic settings
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#e6edf3',
    'xtick.color':      '#e6edf3',
    'ytick.color':      '#e6edf3',
    'text.color':       '#e6edf3',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'grid.alpha':       0.6,
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})
PALETTE = ['#58a6ff', '#f78166', '#3fb950', '#d2a8ff']
print(" Libraries loaded. Environment configured.")


 Libraries loaded. Environment configured.


In [ ]:
# Download cleaned_master.csv directly from GitHub if not already present
import os
import urllib.request

DATA_URL  = "https://raw.githubusercontent.com/lanoboi/VCT-Player-Decay/main/aggregated/cleaned_master.csv"
DATA_PATH = "cleaned_master.csv"

if not os.path.exists(DATA_PATH):
    print("Downloading cleaned_master.csv from GitHub...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print("Done.")
else:
    print("cleaned_master.csv already present.")

Done.


In [ ]:

# 2.4 Data Loading
df_raw = pd.read_csv('cleaned_master.csv')
print(f"Raw dataset: {df_raw.shape[0]} matches × {df_raw.shape[1]} columns")
print(f"\nColumn groups:")
print(f"  Metadata  : {['Tournament','Stage','Match Type','Match Name','Team A','Team B','Match ID']}")
print(f"  Team A KPIs: {[c for c in df_raw.columns if c.startswith('ta_') and 'hist' not in c]}")
print(f"  Team B KPIs: {[c for c in df_raw.columns if c.startswith('tb_') and 'hist' not in c]}")
print(f"  Target    : team_a_won")
print(f"\nClass distribution:")
vc = df_raw['team_a_won'].value_counts()
print(f"  Team A wins (y=1): {vc[1]:4d}  ({100*vc[1]/len(df_raw):.1f}%)")
print(f"  Team B wins (y=0): {vc[0]:4d}  ({100*vc[0]/len(df_raw):.1f}%)")


Raw dataset: 1265 matches × 47 columns

Column groups:
  Metadata  : ['Tournament', 'Stage', 'Match Type', 'Match Name', 'Team A', 'Team B', 'Match ID']
  Team A KPIs: ['ta_avg_rating', 'ta_avg_acs', 'ta_avg_kd', 'ta_avg_adr', 'ta_avg_fk', 'ta_avg_hs', 'ta_eco_$ (won)', 'ta_eco_$$ (won)', 'ta_eco_$$$ (won)', 'ta_eco_Eco (won)', 'ta_eco_Pistol Won', 'ta_total_2k', 'ta_total_3k', 'ta_total_4k', 'ta_total_5k', 'ta_total_clutches', 'ta_draft_ban', 'ta_draft_pick']
  Team B KPIs: ['tb_avg_rating', 'tb_avg_acs', 'tb_avg_kd', 'tb_avg_adr', 'tb_avg_fk', 'tb_avg_hs', 'tb_eco_$ (won)', 'tb_eco_$$ (won)', 'tb_eco_$$$ (won)', 'tb_eco_Eco (won)', 'tb_eco_Pistol Won', 'tb_total_2k', 'tb_total_3k', 'tb_total_4k', 'tb_total_5k', 'tb_total_clutches', 'tb_draft_ban', 'tb_draft_pick']
  Target    : team_a_won

Class distribution:
  Team A wins (y=1):  664  (52.5%)
  Team B wins (y=0):  601  (47.5%)


In [ ]:

# 2.5 Season Assignment
df = df_raw.copy()

def assign_season(tournament_name: str) -> int:
    """Assign a VCT season year based on the tournament name."""
    if '2023' in tournament_name:
        return 2023
    elif '2024' in tournament_name:
        return 2024
    return 2025

df['season'] = df['Tournament'].apply(assign_season)

season_counts = df['season'].value_counts().sort_index()
print("Match counts per season (dataset):")
for yr, cnt in season_counts.items():
    print(f"  D{yr-2022} - {yr}: {cnt} matches")


Match counts per season (dataset):
  D1 - 2023: 330 matches
  D2 - 2024: 434 matches
  D3 - 2025: 501 matches


In [ ]:

# 2.6 Context Feature Engineering

# Binary: is this an elimination/must-win match?
ELIM_KEYWORDS = {'elimination', 'lower', 'decider'}
df['is_elimination_match'] = df['Match Type'].str.lower().apply(
    lambda mt: int(any(kw in mt for kw in ELIM_KEYWORDS))
)

# Binary: is this the Grand Final?
df['is_grand_final'] = (
    df['Match Type'].str.strip().str.lower() == 'grand final'
).astype(int)

# Ordinal: match importance (1=league, 2=group/knockout, 3=playoff/finals)
def stage_stakes_score(stage: str) -> int:
    s = stage.lower()
    if any(kw in s for kw in {'playoff', 'bracket', 'final', 'play-in'}):
        return 3
    elif any(kw in s for kw in {'group', 'swiss', 'knockout', 'main event', 'play-ins'}):
        return 2
    return 1

df['stage_stakes'] = df['Stage'].apply(stage_stakes_score)

# Binary: Team A holds draft ban slot #1
df['ta_ban_first'] = (df['ta_draft_ban'] == 1).astype(int)

print("Context features engineered:")
print(f"  is_elimination_match: {df['is_elimination_match'].sum()} elimination matches")
print(f"  is_grand_final      : {df['is_grand_final'].sum()} grand finals")
print(f"  stage_stakes dist   : {dict(df['stage_stakes'].value_counts().sort_index())}")
print(f"  ta_ban_first        : {df['ta_ban_first'].sum()} matches where Team A bans first")


Context features engineered:
  is_elimination_match: 216 elimination matches
  is_grand_final      : 40 grand finals
  stage_stakes dist   : {1: np.int64(391), 2: np.int64(508), 3: np.int64(366)}
  ta_ban_first        : 46 matches where Team A bans first


In [ ]:

# 2.7 Historical Rolling Feature Engineering (Zero-Leakage)
# Rolling historical features computed with strict temporal integrity.
# Accumulators are updated AFTER each match's feature vector is recorded.

# Sort chronologically before iterating
df = df.sort_values(['season', 'Match ID']).reset_index(drop=True)

# Per-team accumulators
team_wins         = defaultdict(list)   # list of 0/1 outcomes per prior match
team_ratings      = defaultdict(list)   # list of avg_rating per prior match
team_maps_won     = defaultdict(int)    # cumulative maps won
team_maps_played  = defaultdict(int)    # cumulative maps played (both teams' maps)

MIN_MATCHES   = 3      # minimum prior matches before trusting historical stats
PRIOR_WR      = 0.50   # uninformative win-rate prior
PRIOR_RATING  = 1.00   # Valorant rating baseline (~1.0 for average player)
PRIOR_MAP_PCT = 0.50   # uninformative map win percentage prior

# Storage arrays (populated in order, then attached as columns)
ta_hist_win_rate_arr    = []
tb_hist_win_rate_arr    = []
ta_hist_avg_rating_arr  = []
tb_hist_avg_rating_arr  = []
ta_hist_map_win_pct_arr = []
tb_hist_map_win_pct_arr = []

for _, row in df.iterrows():
    ta, tb = row['Team A'], row['Team B']

    # PRE-MATCH LOOKUP (features)
    ta_wr = np.mean(team_wins[ta])        if len(team_wins[ta])   >= MIN_MATCHES else PRIOR_WR
    tb_wr = np.mean(team_wins[tb])        if len(team_wins[tb])   >= MIN_MATCHES else PRIOR_WR
    ta_ar = np.mean(team_ratings[ta])     if len(team_ratings[ta])>= MIN_MATCHES else PRIOR_RATING
    tb_ar = np.mean(team_ratings[tb])     if len(team_ratings[tb])>= MIN_MATCHES else PRIOR_RATING
    ta_mp = (team_maps_won[ta] / team_maps_played[ta]
             if team_maps_played[ta] >= MIN_MATCHES else PRIOR_MAP_PCT)
    tb_mp = (team_maps_won[tb] / team_maps_played[tb]
             if team_maps_played[tb] >= MIN_MATCHES else PRIOR_MAP_PCT)

    ta_hist_win_rate_arr.append(ta_wr);    tb_hist_win_rate_arr.append(tb_wr)
    ta_hist_avg_rating_arr.append(ta_ar);  tb_hist_avg_rating_arr.append(tb_ar)
    ta_hist_map_win_pct_arr.append(ta_mp); tb_hist_map_win_pct_arr.append(tb_mp)

    # POST-MATCH UPDATE (outcomes update accumulators AFTER feature recording)
    ta_won = int(row['team_a_won'])
    team_wins[ta].append(ta_won);        team_wins[tb].append(1 - ta_won)
    team_ratings[ta].append(row['ta_avg_rating'])
    team_ratings[tb].append(row['tb_avg_rating'])
    team_maps_won[ta]    += int(row['Team A Score'])
    team_maps_won[tb]    += int(row['Team B Score'])
    team_maps_played[ta] += int(row['Team A Score']) + int(row['Team B Score'])
    team_maps_played[tb] += int(row['Team A Score']) + int(row['Team B Score'])

# Attach historical features to dataframe
df['ta_hist_win_rate']    = ta_hist_win_rate_arr
df['tb_hist_win_rate']    = tb_hist_win_rate_arr
df['ta_hist_avg_rating']  = ta_hist_avg_rating_arr
df['tb_hist_avg_rating']  = tb_hist_avg_rating_arr
df['ta_hist_map_win_pct'] = ta_hist_map_win_pct_arr
df['tb_hist_map_win_pct'] = tb_hist_map_win_pct_arr

# Differential features (capture relative strength between the two teams)
df['hist_win_rate_diff']  = df['ta_hist_win_rate']    - df['tb_hist_win_rate']
df['hist_rating_diff']    = df['ta_hist_avg_rating']  - df['tb_hist_avg_rating']
df['map_win_pct_diff']    = df['ta_hist_map_win_pct'] - df['tb_hist_map_win_pct']

print(" Historical features engineered with zero data leakage.")
print(f"   All 13 model features now present in dataframe.")


 Historical features engineered with zero data leakage.
   All 13 model features now present in dataframe.


In [ ]:
# 2.8 Walk-Forward Validation Setup
# Walk-Forward folds

# Fold 1: Train on 2023          -> Test on 2024   (single-season training)
# Fold 2: Train on 2023 + 2024   -> Test on 2025   (expanding window)

FEATURES = [
    'ta_hist_win_rate',    'tb_hist_win_rate',    'hist_win_rate_diff',
    'ta_hist_avg_rating',  'tb_hist_avg_rating',  'hist_rating_diff',
    'ta_hist_map_win_pct', 'tb_hist_map_win_pct', 'map_win_pct_diff',
    'stage_stakes', 'is_elimination_match', 'is_grand_final', 'ta_ban_first'
]
TARGET = 'team_a_won'

# Walk-Forward fold definitions
FOLDS = [
    {
        'label':         'Fold 1',
        'description':   'Train: 2023  ->  Test: 2024',
        'train_seasons': [2023],
        'test_season':   2024,
    },
    {
        'label':         'Fold 2',
        'description':   'Train: 2023 + 2024  ->  Test: 2025',
        'train_seasons': [2023, 2024],
        'test_season':   2025,
    },
]

# Fold-size summary
print("=" * 62)
print("  WALK-FORWARD VALIDATION - FOLD STRUCTURE")
print("=" * 62)
for i, fold in enumerate(FOLDS, start=1):
    tr = df[df['season'].isin(fold['train_seasons'])]
    te = df[df['season'] == fold['test_season']]
    tr_wr = df.loc[tr.index, TARGET].mean()
    te_wr = df.loc[te.index, TARGET].mean()
    print(f"\n  {fold['label']} - {fold['description']}")
    print(f"    Training : {len(tr):4d} matches  (Team-A win rate: {tr_wr:.3f})")
    print(f"    Test     : {len(te):4d} matches  (Team-A win rate: {te_wr:.3f})")
print()
print("  Rationale: Walk-forward validation preserves strict temporal order.")
print("  The scaler is re-fitted on each fold's training split only, so no")
print("  future-season statistics ever influence feature normalisation.")

# Primary (Fold-2) split
train_df = df[df['season'].isin([2023, 2024])].copy()
test_df  = df[df['season'] == 2025].copy()

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test,  y_test  = test_df[FEATURES],  test_df[TARGET]

# Scaler for the primary (Fold-2) split -- used by Section 5 onwards
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit ONLY on training data
X_test_sc  = scaler.transform(X_test)        # apply same transform to test

print(f"\nWalk-forward folds defined.  Primary split (Fold 2) also ready for")
print(f"   Section 5 training.  Feature dimensions: {len(FEATURES)}")


  WALK-FORWARD VALIDATION - FOLD STRUCTURE

  Fold 1 - Train: 2023  ->  Test: 2024
    Training :  330 matches  (Team-A win rate: 0.512)
    Test     :  434 matches  (Team-A win rate: 0.539)

  Fold 2 - Train: 2023 + 2024  ->  Test: 2025
    Training :  764 matches  (Team-A win rate: 0.527)
    Test     :  501 matches  (Team-A win rate: 0.521)

  Rationale: Walk-forward validation preserves strict temporal order.
  The scaler is re-fitted on each fold's training split only, so no
  future-season statistics ever influence feature normalisation.

Walk-forward folds defined.  Primary split (Fold 2) also ready for
   Section 5 training.  Feature dimensions: 13
